# 03 Do-Intervention Inference — ALARM
The intervention experiment follows a fixed procedure: generic child slicing →  remove VMCH  →  condition on $G_{do}$ →  four-order FGCR →  target query. The current frozen instance is do(VMCH=NORMAL), and the protocol asserts that$\operatorname{ch}_G(VMCH)=\{VTUB\}$
The generic intervention function itself does not hard-code VTUB.

## 1. Load the Shared Core from Notebook 00


In [1]:
import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
COMMON_CORE_PATH = PROJECT_ROOT / "00_FGCR_Common_Core.ipynb"
if not COMMON_CORE_PATH.exists():
    raise FileNotFoundError("Missing 00_FGCR_Common_Core.ipynb in the project root")

_common_notebook = json.loads(COMMON_CORE_PATH.read_text(encoding="utf-8"))
_loaded_common_cells = 0
for _cell in _common_notebook["cells"]:
    _tags = _cell.get("metadata", {}).get("tags", [])
    if _cell.get("cell_type") == "code" and "common_core" in _tags:
        exec(compile("".join(_cell.get("source", [])), str(COMMON_CORE_PATH), "exec"), globals())
        _loaded_common_cells += 1

if _loaded_common_cells != 7:
    raise RuntimeError(f"Expected 7 common_core cells, loaded {_loaded_common_cells}")


## 2. Core Computation for Post-Intervention Conditional Inference


In [2]:
RUN_MODE = os.environ.get("FGCR_RUN_MODE", "formal").strip().lower()
EXACT_TOLERANCE = 1e-10

# Convert a target-array index into a state-name assignment for the full target-distribution result table.
def do_target_cell_assignment(index: tuple[int, ...]) -> dict[Node, str]:
    return {
        node: INDEX_TO_STATE[node][state_index]
        for node, state_index in zip(TARGET, index)
    }

# Store all target-state cells and cell-wise errors for one post-intervention method.
def append_do_distribution_cells(
    rows: list[dict[str, object]],
    *,
    parameter_id: int,
    method: str,
    order_id: str,
    truth: np.ndarray,
    estimate: np.ndarray,
    protocol_sha256: str,
) -> None:
    for index in np.ndindex(truth.shape):
        rows.append({
            "parameter_id": parameter_id,
            "method": method,
            "order_id": order_id,
            "target_assignment_json": json.dumps(
                do_target_cell_assignment(index), sort_keys=True
            ),
            "p_true_do": float(truth[index]),
            "p_estimate_do": float(estimate[index]),
            "absolute_error_do": float(abs(estimate[index] - truth[index])),
            "protocol_sha256": protocol_sha256,
        })

# Run the known-parameter exact experiment after the deterministic intervention.
def run_experiment3(
    project_root: Path,
    run_mode: str,
    experiment_code_sha256: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, str]:
    project_root = Path(project_root)
    settings = get_settings(run_mode)
    bank = ensure_shared_parameter_bank(project_root / "data" / "alarm_parameter_sets.xlsx")
    selected_ids = tuple(int(value) for value in settings["parameter_ids"])

    # Structural identity includes the intervention graph, the four H_R_do graphs, and G_R_do used by IC.
    structure_hashes = {
        "G_do": edge_sha256(G_DO),
        **{
            f"FGCR_do_{order_id}": edge_sha256(result.H_R)
            for order_id, result in INTERVENTIONAL_STRUCTURES.items()
        },
        "IC_G_R_do": edge_sha256(G_R_DO),
    }
    experiment_payload = {
        "intervention": INTERVENTION,
        "transform": FROZEN_PROTOCOL["intervention"]["transform"],
        "expected_children": EXPECTED_INTERVENTION_CHILDREN,
        "condition": CONDITION,
        "target": TARGET,
        "target_event": TARGET_EVENT,
        "truth": "post_intervention_ancestral_enumeration_checked_by_VE",
        "exact_tolerance": EXACT_TOLERANCE,
        "order_difference": "maximum pairwise TVD among four do-FGCR target distributions",
    }
    protocol_sha256 = experiment_protocol_sha256(
        3,
        run_mode,
        bank.content_sha256,
        structure_hashes,
        experiment_payload,
        experiment_code_sha256,
    )

    run_rows: list[dict[str, object]] = []
    cell_rows: list[dict[str, object]] = []
    order_rows: list[dict[str, object]] = []

    for parameter_id in selected_ids:
        source_cpts = bank.cpts(parameter_id)
        parameter_sha256 = cpts_sha256(source_cpts, PARENT_ORDER)

        # For every parameter set, call the same generic function: automatically identify child nodes, slice parent axes, and remove VMCH.
        do_model = intervene_and_remove(
            ALARM_DAG, source_cpts, PARENT_ORDER, INTERVENTION
        )
        if do_model.graph.edges != G_DO.edges:
            raise AssertionError("Post-intervention graph differs from frozen G_DO")
        if do_model.sliced_children != EXPECTED_INTERVENTION_CHILDREN:
            raise AssertionError("Observed intervention children differ from protocol")

        #  The independent post-intervention enumeration oracle and VE must agree before entering the FGCR comparison.
        truth_do = ancestral_enumeration_query(
            do_model.cpts,
            do_model.graph,
            do_model.parent_order,
            TARGET,
            CONDITION,
        )
        truth_do_ve = ve_query(
            do_model.cpts, do_model.parent_order, TARGET, CONDITION
        )
        oracle_errors = distribution_errors(truth_do, truth_do_ve)
        if max(oracle_errors.values()) > EXACT_TOLERANCE:
            raise AssertionError(f"do oracle/VE mismatch for parameter {parameter_id}")
        p_true_do = event_probability(truth_do, TARGET, TARGET_EVENT)

        condition_distribution_do = ve_query(
            do_model.cpts, do_model.parent_order, CONDITION_NODES
        )
        p_condition_do = event_probability(
            condition_distribution_do, CONDITION_NODES, CONDITION
        )

        #  do-J is an independent oracle on the intervention graph; explicitly store a zero-error baseline row.
        run_rows.append({
            "experiment_id": 3,
            "run_mode": run_mode,
            "parameter_id": parameter_id,
            "method": "J",
            "order_id": "none",
            "intervention_json": json.dumps(INTERVENTION, sort_keys=True),
            "sliced_children_json": json.dumps(do_model.sliced_children, sort_keys=True),
            "graph_source": "G_do_independent_oracle",
            "edge_count": len(G_DO.edges),
            "parameter_dimension": parameter_dimension(do_model.parent_order),
            "p_true_do": p_true_do,
            "p_estimate_do": p_true_do,
            "error_do": 0.0,
            "absolute_error_do": 0.0,
            "relative_bias_do": 0.0,
            "tvd_do": 0.0,
            "max_abs_error_do": 0.0,
            "condition_probability_do": p_condition_do,
            "parameter_sha256": parameter_sha256,
            "parameter_bank_sha256": bank.content_sha256,
            "structure_sha256": edge_sha256(G_DO),
            "protocol_sha256": protocol_sha256,
        })
        append_do_distribution_cells(
            cell_rows,
            parameter_id=parameter_id,
            method="J",
            order_id="none",
            truth=truth_do,
            estimate=truth_do,
            protocol_sha256=protocol_sha256,
        )

        # IC must run on G_R_do obtained after removing the conditioned nodes from G_do; it must not revert to the observational G_R.
        ic_cpts, ic_parent_order = compile_conditional_cpts(
            do_model.cpts,
            do_model.parent_order,
            G_R_DO,
            CONDITION,
        )
        ic_distribution = ve_query(ic_cpts, ic_parent_order, TARGET)
        ic_metrics = distribution_errors(truth_do, ic_distribution)
        p_ic_do = event_probability(ic_distribution, TARGET, TARGET_EVENT)
        ic_error = p_ic_do - p_true_do
        run_rows.append({
            "experiment_id": 3,
            "run_mode": run_mode,
            "parameter_id": parameter_id,
            "method": "IC",
            "order_id": "none",
            "intervention_json": json.dumps(INTERVENTION, sort_keys=True),
            "sliced_children_json": json.dumps(do_model.sliced_children, sort_keys=True),
            "graph_source": "G_do_then_induced_G_R_do",
            "edge_count": len(G_R_DO.edges),
            "parameter_dimension": parameter_dimension(G_R_DO),
            "p_true_do": p_true_do,
            "p_estimate_do": p_ic_do,
            "error_do": ic_error,
            "absolute_error_do": abs(ic_error),
            "relative_bias_do": safe_relative_bias(ic_error, p_true_do),
            "tvd_do": ic_metrics["tvd"],
            "max_abs_error_do": ic_metrics["max_abs_error"],
            "condition_probability_do": p_condition_do,
            "parameter_sha256": parameter_sha256,
            "parameter_bank_sha256": bank.content_sha256,
            "structure_sha256": edge_sha256(G_R_DO),
            "protocol_sha256": protocol_sha256,
        })
        append_do_distribution_cells(
            cell_rows,
            parameter_id=parameter_id,
            method="IC",
            order_id="none",
            truth=truth_do,
            estimate=ic_distribution,
            protocol_sha256=protocol_sha256,
        )

        fgcr_event_values: list[float] = []
        fgcr_distributions: list[np.ndarray] = []
        for order_id, result in INTERVENTIONAL_STRUCTURES.items():
            # For each H_R_do, compile local CPTs exactly from the same P_do and then perform the conditional query independently.
            hr_cpts, hr_parent_order = compile_conditional_cpts(
                do_model.cpts,
                do_model.parent_order,
                result.H_R,
                CONDITION,
            )
            estimate = ve_query(hr_cpts, hr_parent_order, TARGET)
            metrics = distribution_errors(truth_do, estimate)
            p_estimate = event_probability(estimate, TARGET, TARGET_EVENT)
            error = p_estimate - p_true_do
            if max(metrics.values()) > EXACT_TOLERANCE or abs(error) > EXACT_TOLERANCE:
                raise AssertionError(
                    f"Exact do-FGCR failed: parameter={parameter_id}, order={order_id}"
                )
            fgcr_event_values.append(p_estimate)
            fgcr_distributions.append(estimate)
            run_rows.append({
                "experiment_id": 3,
                "run_mode": run_mode,
                "parameter_id": parameter_id,
                "method": "FGCR-C",
                "order_id": order_id,
                "intervention_json": json.dumps(INTERVENTION, sort_keys=True),
                "sliced_children_json": json.dumps(do_model.sliced_children, sort_keys=True),
                "graph_source": "G_do_to_FGCR_generated_H_R_do",
                "beta_do": ",".join(result.orders.beta),
                "alpha_C": ",".join(result.orders.alpha_C),
                "alpha_R_do": ",".join(result.orders.alpha_R),
                "edge_count": len(result.H_R.edges),
                "fill_edge_count": len(result.added_edges_R),
                "parameter_dimension": parameter_dimension(result.H_R),
                "p_true_do": p_true_do,
                "p_estimate_do": p_estimate,
                "error_do": error,
                "absolute_error_do": abs(error),
                "relative_bias_do": safe_relative_bias(error, p_true_do),
                "tvd_do": metrics["tvd"],
                "max_abs_error_do": metrics["max_abs_error"],
                "condition_probability_do": p_condition_do,
                "parameter_sha256": parameter_sha256,
                "parameter_bank_sha256": bank.content_sha256,
                "structure_sha256": edge_sha256(result.H_R),
                "protocol_sha256": protocol_sha256,
            })
            append_do_distribution_cells(
                cell_rows,
                parameter_id=parameter_id,
                method="FGCR-C",
                order_id=order_id,
                truth=truth_do,
                estimate=estimate,
                protocol_sha256=protocol_sha256,
            )

        # For the intervention experiment, OD likewise uses the maximum pairwise TVD over the complete target distributions.
        order_tvd = max(
            distribution_errors(first, second)["tvd"]
            for first, second in combinations(fgcr_distributions, 2)
        )
        event_difference = max(fgcr_event_values) - min(fgcr_event_values)
        if order_tvd > EXACT_TOLERANCE or event_difference > EXACT_TOLERANCE:
            raise AssertionError(f"do-FGCR order difference exceeded tolerance: {parameter_id}")
        order_rows.append({
            "parameter_id": parameter_id,
            "order_difference_tvd_do": order_tvd,
            "max_minus_min_event_probability_do": event_difference,
            "passed_le_1e_10": order_tvd <= EXACT_TOLERANCE,
        })

    runs = pd.DataFrame(run_rows)
  # The graph_source field for both methods must use G_do as its semantic starting point.
    if not runs["graph_source"].str.contains("G_do").all():
        raise AssertionError("Experiment 3 contains a non-interventional graph source")
    distribution_cells = pd.DataFrame(cell_rows)
    order_differences = pd.DataFrame(order_rows)
    summary = (
        runs.groupby(["method", "order_id"], sort=False)
        .agg(
            n_parameter_sets=("parameter_id", "nunique"),
            mean_absolute_error_do=("absolute_error_do", "mean"),
            max_absolute_error_do=("absolute_error_do", "max"),
            mean_tvd_do=("tvd_do", "mean"),
            max_tvd_do=("tvd_do", "max"),
            mean_max_cell_error_do=("max_abs_error_do", "mean"),
            max_max_cell_error_do=("max_abs_error_do", "max"),
            mean_relative_bias_do=("relative_bias_do", "mean"),
        )
        .reset_index()
    )
    protocol = experiment_protocol_frame(
        3,
        run_mode,
        bank.content_sha256,
        structure_hashes,
        experiment_payload,
        experiment_code_sha256,
        protocol_sha256,
    )
    return (
        runs,
        distribution_cells,
        order_differences,
        summary,
        protocol,
        protocol_sha256,
    )


## 3. Output and Plotting


In [3]:
# Save the post-intervention repetition-level results, all target states, the frozen do-structure, and the experiment protocol.
def save_experiment3(
    project_root: Path,
    runs: pd.DataFrame,
    distribution_cells: pd.DataFrame,
    order_differences: pd.DataFrame,
    summary: pd.DataFrame,
    protocol: pd.DataFrame,
    protocol_sha256: str,
) -> Path:
    output = Path(project_root) / "result" / "03_do_intervention_inference.xlsx"
    write_excel_atomic(output, {
        "runs": runs,
        "distribution_cells": distribution_cells,
        "order_difference": order_differences,
        "summary": summary,
        "structures": INTERVENTIONAL_STRUCTURE_AUDIT,
        "protocol": protocol,
        "environment": pd.DataFrame([environment_record()]),
    })
    verify_output_protocol(output, protocol_sha256)
    return output

#  Plot the target-event absolute errors for the four-order do-FGCR methods and do-IC.
def plot_experiment3(project_root: Path, runs: pd.DataFrame) -> Path:
    os.environ.setdefault("MPLCONFIGDIR", str(Path(project_root) / "result" / "matplotlib_cache"))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    output = Path(project_root) / "result" / "03_do_intervention_absolute_error.png"
    # In the intervention experiment, J-Method provides the exact post-intervention conditional truth and is therefore excluded from the error-comparison plot.
    labels = [ f"C-Method_{index}" for index in range(1, 5)] + ["IC"]

    values = [ runs.loc[ (runs["method"] == "FGCR-C")& (runs["order_id"] == f"order_{index}"), "absolute_error_do",].to_numpy(float)
        for index in range(1, 5)] + [ runs.loc[ runs["method"] == "IC", "absolute_error_do",].to_numpy(float) ]

  #  plot_floor = np.finfo(float).tiny
    # Absolute probability error is at most 1; use 1e-20 only to display zero errors on the logarithmic scale without modifying the original results.
    plot_floor = 1e-20
    fig, ax = plt.subplots(figsize=(9.2, 5.0), constrained_layout=True)
    rng = np.random.default_rng(20260826)
    for x, method_values in enumerate(values, 1):
        shown = np.maximum(method_values, plot_floor)
        jitter = rng.uniform(-0.10, 0.10, len(shown))
        ax.scatter(np.full(len(shown), x) + jitter, shown, s=22, alpha=0.55)
        ax.hlines(np.median(shown), x - 0.18, x + 0.18, linewidth=2.2)
    ax.axhline(EXACT_TOLERANCE, linestyle="--", linewidth=1.1, label=r"$10^{-10}$ tolerance")
    ax.set_yscale("log")
    # Fix a reasonable display range so that zero errors do not expand the vertical axis down to approximately 1e-308.
    ax.set_ylim(plot_floor, 1.0)
    ax.set_xticks(range(1, len(labels) + 1), labels)
    ax.set_ylabel("Absolute error of target-event probability")
    ax.set_title("Conditional inference after do(VMCH=NORMAL)")
    ax.legend(frameon=False)
    fig.savefig(output, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return output


## 4. Execution


In [4]:
EXPERIMENT3_CODE_SHA256 = tagged_code_sha256(
    PROJECT_ROOT / "03_Do_Intervention_Inference.ipynb", "experiment3_core"
)
(
    experiment3_runs,
    experiment3_distribution_cells,
    experiment3_order_differences,
    experiment3_summary,
    experiment3_protocol,
    experiment3_protocol_sha256,
) = run_experiment3(PROJECT_ROOT, RUN_MODE, EXPERIMENT3_CODE_SHA256)
experiment3_workbook = save_experiment3(
    PROJECT_ROOT,
    experiment3_runs,
    experiment3_distribution_cells,
    experiment3_order_differences,
    experiment3_summary,
    experiment3_protocol,
    experiment3_protocol_sha256,
)
experiment3_figure = plot_experiment3(PROJECT_ROOT, experiment3_runs)
print(experiment3_summary.to_string(index=False))
print(f"Workbook: {experiment3_workbook}")
print(f"Figure:   {experiment3_figure}")


method order_id  n_parameter_sets  mean_absolute_error_do  max_absolute_error_do  mean_tvd_do   max_tvd_do  mean_max_cell_error_do  max_max_cell_error_do  mean_relative_bias_do
     J     none               100            0.000000e+00           0.000000e+00 0.000000e+00 0.000000e+00            0.000000e+00           0.000000e+00           0.000000e+00
    IC     none               100            2.853282e-03           1.984532e-02 2.844224e-02 1.381858e-01            9.521926e-03           4.887292e-02           3.584116e-02
FGCR-C  order_1               100            2.022938e-17           2.220446e-16 2.182101e-16 4.024558e-16            1.487005e-16           4.440892e-16           1.023018e-17
FGCR-C  order_2               100            2.541126e-17           2.220446e-16 2.178718e-16 4.085274e-16            1.458555e-16           3.885781e-16           4.347207e-17
FGCR-C  order_3               100            2.135438e-17           2.220446e-16 2.191072e-16 4.117258e-16         

In [7]:
# from pathlib import Path
# import pandas as pd

# PROJECT_ROOT = Path.cwd().resolve()
# workbook_path = PROJECT_ROOT / "result" / "03_do_intervention_inference.xlsx"

# runs = pd.read_excel(workbook_path, sheet_name="runs")  

# figure_path = plot_experiment3(PROJECT_ROOT, runs)
# print(f"Updated figure: {figure_path}")

Updated figure: D:\jupyternoteboook\FCGR算法实验\条件概率实验\条件概率实验（alarm)\result\03_do_intervention_absolute_error.png
